Use heart.csv to make predictions on heart disease 'AHD'. Apply Logistic Regression.

Notes:
  1. Use standard scaler
  2. Logistic regression can handle categorical target variables (y), but they are typically converted to numerical values. All feature variables (X) must be converted to numerical values. Hence, preprocess the data well.

  GOODLUCK!

In [ ]:
import pandas as pd

df = pd.read_csv("Heart.csv")
df.head()

,Unnamed: 0,Age,Sex,ChestPain,RestBP,Chol,Fbs,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal,AHD
0,1,63,1,typical,145,233,1,2,150,0,2.3,3,0.0,fixed,No
1,2,67,1,asymptomatic,160,286,0,2,108,1,1.5,2,3.0,normal,Yes
2,3,67,1,asymptomatic,120,229,0,2,129,1,2.6,2,2.0,reversable,Yes
3,4,37,1,nonanginal,130,250,0,0,187,0,3.5,3,0.0,normal,No
4,5,41,0,nontypical,130,204,0,2,172,0,1.4,1,0.0,normal,No


In [ ]:
import numpy as np

df = df.replace("NA", np.nan)

In [ ]:
# Separate X and y. Convert target AHD to 0 and 1
y = df["AHD"].map({"No": 0, "Yes": 1})
X = df.drop(columns=["AHD"])

In [ ]:
# Identify which columns are categorical vs numeric
categorical_cols = ["ChestPain", "Thal"]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

In [ ]:
# Build preprocessing. Impute missing. One-hot encode text. Scale numbers

# Numeric pipeline:
#   Fill missing numbers with the median.
#   Scale them so they are on similar ranges.
# Categorical pipeline:
#   Fill missing text with the most common value.
#   One-hot encode. This turns words into 0 and 1 columns.
# ColumnTransformer applies the right steps to the right columns.

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [ ]:
# Split into train and test sets. 80% train, 20% test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Build and train Logistic Regression model using a pipeline
from sklearn.linear_model import LogisticRegression

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000))
])

clf.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Unnamed: 0', 'Age', 'Sex',
                                                   'RestBP', 'Chol', 'Fbs',
                                                   'RestECG', 'MaxHR', 'ExAng',
                                                   'Oldpeak', 'Slope', 'Ca']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['ChestPain', 'Thal'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [ ]:
# Predict on train and test. Then evaluate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

y_train_pred = clf.predict(X_train)
y_test_pred = clf.predict(X_test)

y_train_proba = clf.predict_proba(X_train)[:, 1]
y_test_proba = clf.predict_proba(X_test)[:, 1]

In [ ]:
# Print metrics for both train and test
def print_classification_metrics(name, y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_proba)
    cm = confusion_matrix(y_true, y_pred)

    print(f"{name} set evaluation:")
    print(f"Accuracy:  {acc:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"Recall:    {rec:.3f}")
    print(f"F1:        {f1:.3f}")
    print(f"ROC AUC:   {auc:.3f}")
    print("Confusion matrix:")
    print(cm)
    print()

print_classification_metrics("Train", y_train, y_train_pred, y_train_proba)
print_classification_metrics("Test", y_test, y_test_pred, y_test_proba)

Train set evaluation:
Accuracy:  0.868
Precision: 0.883
Recall:    0.820
F1:        0.850
ROC AUC:   0.929
Confusion matrix:
[[119  12]
 [ 20  91]]

Test set evaluation:
Accuracy:  0.885
Precision: 0.839
Recall:    0.929
F1:        0.881
ROC AUC:   0.960
Confusion matrix:
[[28  5]
 [ 2 26]]



In [ ]:
# Accuracy: overall correct rate.
# Precision: when we say “Yes”, how often we are right.
# Recall: how many real “Yes” cases we catch.
# F1: balance of precision and recall.
# ROC AUC: ranking quality using probabilities.
# Confusion matrix: shows TP, FP, TN, FN.

In [ ]:
# The model did well.
# It means it is not just memorizing training data.
# It's learning patterns that work on new data.

# Train Set Results

- **Accuracy:** 0.868
  - Out of 100 people in the training data, approximately 87 are correctly identified.

- **Precision:** 0.883
  - When the model predicts "Yes, heart disease," it is correct about 88% of the time.

- **Recall:** 0.820
  - Out of all individuals who truly have heart disease, the model detects about 82%.
  - Some cases are missed.

- **F1 Score:** 0.850
  - A balanced score that combines precision and recall.
  - A value of 0.85 indicates strong performance.

- **ROC AUC:** 0.929
  - Measures how well the model separates "Yes" vs "No" cases.
  - A score of 0.93 is considered excellent.

## Confusion Matrix
- **119** true “No” predicted correctly
- **12** false alarms (predicted Yes but actually No)
- **20** missed cases (predicted No but actually Yes) *this is the scary one medically*
- **91** true “Yes” predicted correctly

# Test Set Results

- **Accuracy:** 0.885

On new unseen data, you get about 89 correct out of 100. Great.

- **Precision:** 0.839

When it says “Yes,” it’s right about 84% of the time. Slightly lower than train.

- **Recall:** 0.929

This is very good. It catches about 93% of real heart disease cases on test.


- **F1 Score:** 0.881

Even stronger balance on test than train.

- **ROC AUC:** 0.960

This is excellent. Strong separation of Yes vs No.

## Confusion Matrix
- 28 correct “No”"
- 5 false alarms (No predicted as Yes)
- 2 missed cases (Yes predicted as No) — very low, which is great
- 26 correct “Yes”

# Performance Evaluation

**Does it perform well?** Yes. Very well.

**Overfitting?** No. Test performance is strong and consistent.

## What type of mistakes does it make?

On test, it makes very few missed heart disease cases (2), which is usually the priority.